# Refine Fact Check Dataset với Qwen và Gemma

Notebook này refine toàn bộ dataset CSV `FinalDataset/claims_merged.csv` với cột `id`, `claim`, `image` bằng local Hugging Face `transformers` trên A100 40GB.

Luồng chạy:
1. Chạy Qwen VLM trước và lưu CSV riêng.
2. Giải phóng GPU.
3. Chạy Gemma VLM và lưu CSV riêng.
4. Gộp kết quả hai model vào một CSV dài có cột `model_name`.

Model khuyến nghị:
- Qwen: `Qwen/Qwen2.5-VL-7B-Instruct`.
- Gemma: `google/gemma-3-12b-it` cho A100 40GB. Nếu gặp OOM hoặc chưa được cấp quyền Gemma trên Hugging Face, đổi thành `google/gemma-3-4b-it`.


## 1. Cài dependency

Chạy cell này trên máy A100 nếu môi trường chưa có các package cần thiết. Gemma là gated model, nên bạn có thể cần `huggingface-cli login` trước khi load model.

In [ ]:
# Uncomment if needed on the A100 machine.
# %pip install -U "transformers>=4.51.0" accelerate torch pillow pandas tqdm
# !huggingface-cli login


## 2. Cấu hình đường dẫn và model

Sửa `INPUT_CSV` trước khi chạy. Bật `SMOKE_TEST = True` để chạy thử vài dòng trước; đổi sang `False` khi gửi job full lên A100.

In [ ]:
from __future__ import annotations

import ast
import gc
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import pipeline

DATASET_ROOT = Path("../FinalDataset")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("FinalDataset")
INPUT_CSV = DATASET_ROOT / "claims_merged.csv"
ID_COLUMN = "id"
TEXT_COLUMN = "claim"
IMAGE_COLUMN = "image"

# Run mode
SMOKE_TEST = True  # True: run a tiny validation pass; False: run the full dataset
SMOKE_ROWS = 3
RUN_QWEN = True
RUN_GEMMA = True

OUTPUT_DIR = Path("refined_outputs_smoke" if SMOKE_TEST else "refined_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
# GEMMA_MODEL_ID = "google/gemma-3-12b-it"  # fallback nhẹ hơn: google/gemma-3-4b-it
GEMMA_MODEL_ID = "google/gemma-3-4b-it"

QWEN_OUTPUT_CSV = OUTPUT_DIR / "refined_qwen.csv"
GEMMA_OUTPUT_CSV = OUTPUT_DIR / "refined_gemma.csv"
COMBINED_OUTPUT_CSV = OUTPUT_DIR / "refined_qwen_gemma_combined.csv"

# Generation and checkpointing
MAX_NEW_TOKENS = 1200 if SMOKE_TEST else 2200
TEMPERATURE = 0.0
SAVE_EVERY = 1 if SMOKE_TEST else 25
RETRY_LIMIT = 3
LIMIT_ROWS = SMOKE_ROWS if SMOKE_TEST else None

print("RUN MODE:", "SMOKE" if SMOKE_TEST else "FULL")
print("LIMIT_ROWS:", LIMIT_ROWS)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

torch.backends.cuda.matmul.allow_tf32 = True
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")


## 3. Prompt và schema output

Prompt bám theo structured output trong `REFINE_OUTPUT_SCHEMA_VI.md`. Vì chạy local `transformers` không ép JSON schema cứng như API, notebook sẽ parse JSON và giữ `raw_output` nếu model trả lỗi format.

In [ ]:
SYSTEM_INSTRUCTION = """
You are an input refiner for a Vietnamese multimodal fact-checking pipeline.
You convert a text claim, and optionally an attached image, into a structured JSON representation for RAG and evidence verification.
Return only valid JSON. Do not add markdown fences or explanations outside the JSON object.
""".strip()

def build_refine_prompt(claim: str, image_provided: bool) -> str:
    return f"""
{SYSTEM_INSTRUCTION}

# Input
- Claim: {claim}
- Image provided: {str(image_provided).lower()}

# Rules
- Write all generated field values in accented Vietnamese.
- Preserve named entities, numbers, dates, quoted text, locations, and distinctive visual details.
- Normalize the claim without changing its meaning.
- Create one best `primary_retrieval_query` for the whole input.
- Split the claim into atomic, verifiable facts.
- Each item in `claim_atoms` must include its own `retrieval_queries` for atom-level RAG.
- If one or more images are provided, describe only directly visible evidence across those images. Do not infer identity, intent, off-image events, or unseen context.
- If multiple images are provided, include observations from all relevant images and keep each observation grounded in visible details.
- If no image is provided, use `visual_observations: []` and `alignment.label: "not_enough_visual_info"`.
- Do not decide whether the claim is true or false.
- Queries are only for finding evidence; do not mention vector databases, embedding models, or retrieval backends.
- Use empty arrays `[]` when information is absent.

# Output JSON shape
{{
  "original_claim": "...",
  "normalized_claim": "...",
  "primary_retrieval_query": "...",
  "image_provided": true,
  "language": "vi",
  "claim_atoms": [
    {{
      "id": "c1",
      "text": "A verifiable atomic claim.",
      "check_type": "entity|event|time|location|number|quote|relation|other",
      "priority": "high|medium|low",
      "retrieval_queries": ["Evidence retrieval query for this atomic claim."]
    }}
  ],
  "visual_observations": [
    {{
      "id": "v1",
      "text": "A direct observation from the image.",
      "visible_evidence": ["visible detail"],
      "confidence": "high|medium|low"
    }}
  ],
  "alignment": {{
    "label": "match|partial_match|mismatch|not_enough_visual_info",
    "text": "Short explanation of the relationship between the claim and the image."
  }},
  "key_entities": {{
    "people": [],
    "organizations": [],
    "locations": [],
    "dates": [],
    "numbers": [],
    "other": []
  }},
  "search_queries": {{
    "semantic": [],
    "keywords": [],
    "visual": []
  }},
  "retrieval_focus": {{
    "text": true,
    "image": true,
    "cross_modal": true
  }},
  "constraints": {{
    "time": [],
    "location": [],
    "source_type": []
  }},
  "context_summary": "...",
  "ambiguity_notes": [],
  "verification_targets": []
}}
""".strip()


## 4. Hàm parse, flatten và xử lý ảnh

Các hàm này giữ notebook đơn giản: parse JSON, chuẩn hóa field thiếu, và flatten object thành cột CSV đủ thông tin cho RAG testing.

In [ ]:
REQUIRED_TOP_LEVEL_KEYS = [
    "original_claim",
    "normalized_claim",
    "primary_retrieval_query",
    "image_provided",
    "language",
    "claim_atoms",
    "visual_observations",
    "alignment",
    "key_entities",
    "search_queries",
    "retrieval_focus",
    "constraints",
    "context_summary",
    "ambiguity_notes",
    "verification_targets",
]

def parse_image_values(value) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple)):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except Exception:
            pass
    if " | " in text:
        return [item.strip() for item in text.split(" | ") if item.strip()]
    return [text]

def resolve_image_paths(value, dataset_root: Path) -> list[Path]:
    paths = []
    for item in parse_image_values(value):
        path = Path(item)
        if not path.is_absolute():
            path = dataset_root / path
        if path.exists():
            paths.append(path)
    return paths

def extract_generated_text(pipe_output) -> str:
    generated = pipe_output[0].get("generated_text", "")
    if isinstance(generated, str):
        return generated.strip()
    if isinstance(generated, list):
        for item in reversed(generated):
            if item.get("role") == "assistant":
                content = item.get("content", "")
                if isinstance(content, str):
                    return content.strip()
                if isinstance(content, list):
                    return "\n".join(str(x.get("text", x)) for x in content).strip()
        return str(generated[-1]).strip() if generated else ""
    return str(generated).strip()

def extract_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))

def normalize_refine_json(data: dict, claim: str, image_provided: bool) -> dict:
    data.setdefault("original_claim", claim)
    data.setdefault("normalized_claim", claim)
    data.setdefault("primary_retrieval_query", data.get("normalized_claim", claim))
    data.setdefault("image_provided", image_provided)
    data.setdefault("language", "vi")
    data.setdefault("claim_atoms", [])
    data.setdefault("visual_observations", [])
    data.setdefault("alignment", {"label": "not_enough_visual_info", "text": "Không đủ thông tin hình ảnh để đối chiếu."})
    data.setdefault("key_entities", {"people": [], "organizations": [], "locations": [], "dates": [], "numbers": [], "other": []})
    data.setdefault("search_queries", {"semantic": [], "keywords": [], "visual": []})
    data.setdefault("retrieval_focus", {"text": True, "image": bool(image_provided), "cross_modal": bool(image_provided)})
    data.setdefault("constraints", {"time": [], "location": [], "source_type": []})
    data.setdefault("context_summary", "")
    data.setdefault("ambiguity_notes", [])
    data.setdefault("verification_targets", [])
    return data

def validate_refine_json(data: dict) -> list[str]:
    errors = []
    for key in REQUIRED_TOP_LEVEL_KEYS:
        if key not in data:
            errors.append(f"missing top-level key: {key}")

    if data.get("language") != "vi":
        errors.append("language must be 'vi'")
    if not isinstance(data.get("primary_retrieval_query"), str) or not data.get("primary_retrieval_query", "").strip():
        errors.append("primary_retrieval_query must be a non-empty string")

    claim_atoms = data.get("claim_atoms")
    if not isinstance(claim_atoms, list):
        errors.append("claim_atoms must be a list")
    else:
        for i, atom in enumerate(claim_atoms):
            if not isinstance(atom, dict):
                errors.append(f"claim_atoms[{i}] must be an object")
                continue
            for key in ["id", "text", "check_type", "priority", "retrieval_queries"]:
                if key not in atom:
                    errors.append(f"claim_atoms[{i}] missing key: {key}")
            if atom.get("check_type") not in ["entity", "event", "time", "location", "number", "quote", "relation", "other"]:
                errors.append(f"claim_atoms[{i}].check_type has invalid value")
            if atom.get("priority") not in ["high", "medium", "low"]:
                errors.append(f"claim_atoms[{i}].priority has invalid value")
            if not isinstance(atom.get("retrieval_queries"), list):
                errors.append(f"claim_atoms[{i}].retrieval_queries must be a list")

    visual_observations = data.get("visual_observations")
    if not isinstance(visual_observations, list):
        errors.append("visual_observations must be a list")
    else:
        for i, obs in enumerate(visual_observations):
            if not isinstance(obs, dict):
                errors.append(f"visual_observations[{i}] must be an object")
                continue
            for key in ["id", "text", "visible_evidence", "confidence"]:
                if key not in obs:
                    errors.append(f"visual_observations[{i}] missing key: {key}")
            if obs.get("confidence") not in ["high", "medium", "low"]:
                errors.append(f"visual_observations[{i}].confidence has invalid value")
            if not isinstance(obs.get("visible_evidence"), list):
                errors.append(f"visual_observations[{i}].visible_evidence must be a list")

    alignment = data.get("alignment")
    if not isinstance(alignment, dict):
        errors.append("alignment must be an object")
    else:
        if alignment.get("label") not in ["match", "partial_match", "mismatch", "not_enough_visual_info"]:
            errors.append("alignment.label has invalid value")
        if not isinstance(alignment.get("text"), str):
            errors.append("alignment.text must be a string")

    key_entities = data.get("key_entities")
    if not isinstance(key_entities, dict):
        errors.append("key_entities must be an object")
    else:
        for key in ["people", "organizations", "locations", "dates", "numbers", "other"]:
            if not isinstance(key_entities.get(key), list):
                errors.append(f"key_entities.{key} must be a list")

    search_queries = data.get("search_queries")
    if not isinstance(search_queries, dict):
        errors.append("search_queries must be an object")
    else:
        for key in ["semantic", "keywords", "visual"]:
            if not isinstance(search_queries.get(key), list):
                errors.append(f"search_queries.{key} must be a list")

    retrieval_focus = data.get("retrieval_focus")
    if not isinstance(retrieval_focus, dict):
        errors.append("retrieval_focus must be an object")
    else:
        for key in ["text", "image", "cross_modal"]:
            if not isinstance(retrieval_focus.get(key), bool):
                errors.append(f"retrieval_focus.{key} must be a boolean")

    constraints = data.get("constraints")
    if not isinstance(constraints, dict):
        errors.append("constraints must be an object")
    else:
        for key in ["time", "location", "source_type"]:
            if not isinstance(constraints.get(key), list):
                errors.append(f"constraints.{key} must be a list")

    for key in ["ambiguity_notes", "verification_targets"]:
        if not isinstance(data.get(key), list):
            errors.append(f"{key} must be a list")
    if not isinstance(data.get("context_summary"), str):
        errors.append("context_summary must be a string")
    return errors

def json_dumps(value) -> str:
    return json.dumps(value, ensure_ascii=False)

def flatten_refine_result(data: dict, model_name: str, raw_output: str, error: str) -> dict:
    alignment = data.get("alignment", {}) or {}
    search_queries = data.get("search_queries", {}) or {}
    retrieval_focus = data.get("retrieval_focus", {}) or {}
    return {
        "model_name": model_name,
        "refined_original_claim": data.get("original_claim", ""),
        "refined_normalized_claim": data.get("normalized_claim", ""),
        "refined_primary_retrieval_query": data.get("primary_retrieval_query", ""),
        "refined_image_provided": data.get("image_provided", False),
        "refined_language": data.get("language", "vi"),
        "refined_claim_atoms": json_dumps(data.get("claim_atoms", [])),
        "refined_visual_observations": json_dumps(data.get("visual_observations", [])),
        "refined_alignment_label": alignment.get("label", ""),
        "refined_alignment_text": alignment.get("text", ""),
        "refined_key_entities": json_dumps(data.get("key_entities", {})),
        "refined_search_semantic": json_dumps(search_queries.get("semantic", [])),
        "refined_search_keywords": json_dumps(search_queries.get("keywords", [])),
        "refined_search_visual": json_dumps(search_queries.get("visual", [])),
        "refined_retrieval_focus_text": retrieval_focus.get("text", False),
        "refined_retrieval_focus_image": retrieval_focus.get("image", False),
        "refined_retrieval_focus_cross_modal": retrieval_focus.get("cross_modal", False),
        "refined_constraints": json_dumps(data.get("constraints", {})),
        "refined_context_summary": data.get("context_summary", ""),
        "refined_ambiguity_notes": json_dumps(data.get("ambiguity_notes", [])),
        "refined_verification_targets": json_dumps(data.get("verification_targets", [])),
        "raw_output": raw_output,
        "refine_error": error,
    }

OUTPUT_COLUMNS = [
    ID_COLUMN,
    TEXT_COLUMN,
    IMAGE_COLUMN,
    "model_name",
    "refined_original_claim",
    "refined_normalized_claim",
    "refined_primary_retrieval_query",
    "refined_image_provided",
    "refined_language",
    "refined_claim_atoms",
    "refined_visual_observations",
    "refined_alignment_label",
    "refined_alignment_text",
    "refined_key_entities",
    "refined_search_semantic",
    "refined_search_keywords",
    "refined_search_visual",
    "refined_retrieval_focus_text",
    "refined_retrieval_focus_image",
    "refined_retrieval_focus_cross_modal",
    "refined_constraints",
    "refined_context_summary",
    "refined_ambiguity_notes",
    "refined_verification_targets",
    "raw_output",
    "refine_error",
]

def write_refine_rows(rows: list[dict], output_csv: Path):
    out = pd.DataFrame(rows)
    for col in OUTPUT_COLUMNS:
        if col not in out.columns:
            out[col] = ""
    out[OUTPUT_COLUMNS].to_csv(output_csv, index=False, encoding="utf-8-sig")


## 5. Smoke preflight trước khi load model

Cell này kiểm tra CSV, cột bắt buộc và một vài đường dẫn ảnh. Chạy cell này trước khi gửi job full lên GPU.

In [ ]:
def preflight_check():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"INPUT_CSV does not exist: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    missing_cols = [col for col in [ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN] if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}; available columns: {list(df.columns)}")

    sample = df.head(LIMIT_ROWS or min(10, len(df))).copy()
    image_status = []
    for idx, row in sample.iterrows():
        raw_path = row.get(IMAGE_COLUMN, "")
        resolved = resolve_image_paths(raw_path, DATASET_ROOT)
        image_status.append({
            "row_index": int(idx),
            "has_claim": bool(str(row.get(TEXT_COLUMN, "")).strip()),
            "raw_image_path": "" if pd.isna(raw_path) else str(raw_path),
            "resolved_image_paths": " | ".join(str(path) for path in resolved),
            "image_count": len(resolved),
        })

    print("Rows in CSV:", len(df))
    print("Rows selected for this run:", len(sample))
    print("Mode:", "SMOKE" if SMOKE_TEST else "FULL")
    display(pd.DataFrame(image_status))
    return df

_ = preflight_check()


## 6. Load model và chạy refine

Notebook dùng `pipeline("image-text-to-text")` để cùng một code path chạy được cả Qwen và Gemma. Nếu bị OOM với Gemma 12B, đổi `GEMMA_MODEL_ID` thành `google/gemma-3-4b-it` và chạy lại phần Gemma.

In [ ]:
def load_vlm(model_id: str):
    return pipeline(
        task="image-text-to-text",
        model=model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

def generate_refine_json(pipe, claim: str, image_paths: list[Path], retry_limit: int = RETRY_LIMIT) -> tuple[dict, str, str]:
    image_provided = bool(image_paths)
    content = []
    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        content.append({"type": "image", "image": image})

    base_prompt = build_refine_prompt(claim, image_provided)
    last_raw = ""
    last_error = ""

    for attempt in range(1, retry_limit + 1):
        attempt_content = list(content)
        prompt = base_prompt
        if attempt > 1:
            prompt += (
                "\n\n# Retry instruction\n"
                f"Your previous output failed validation: {last_error}. "
                "Return a complete valid JSON object that exactly follows the required schema. "
                "Do not omit any required field. Do not add markdown fences."
            )
        attempt_content.append({"type": "text", "text": prompt})
        messages = [{"role": "user", "content": attempt_content}]

        output = pipe(
            messages,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            return_full_text=False,
        )
        last_raw = extract_generated_text(output)
        try:
            parsed = extract_json(last_raw)
            validation_errors = validate_refine_json(parsed)
            if validation_errors:
                last_error = "; ".join(validation_errors[:12])
                continue
            parsed = normalize_refine_json(parsed, claim, image_provided)
            return parsed, last_raw, ""
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"

    fallback = normalize_refine_json({}, claim, image_provided)
    return fallback, last_raw, f"failed validation after {retry_limit} attempts: {last_error}"

def unload_model(pipe):
    del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


## 7. Hàm chạy toàn bộ CSV với resume

Nếu output CSV đã tồn tại, notebook bỏ qua các `id` đã có kết quả cho model đó. Kết quả được lưu sau mỗi `SAVE_EVERY` dòng.

In [ ]:
def run_refinement(model_id: str, output_csv: Path):
    df = pd.read_csv(INPUT_CSV)
    if LIMIT_ROWS is not None:
        df = df.head(LIMIT_ROWS).copy()
    missing_cols = [col for col in [ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN] if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    df = df.reset_index(drop=True)

    if output_csv.exists():
        existing = pd.read_csv(output_csv)
        if ID_COLUMN in existing.columns and "model_name" in existing.columns:
            done_ids = set(existing.loc[existing["model_name"] == model_id, ID_COLUMN].astype(str).tolist())
        else:
            done_ids = set()
        rows = existing.to_dict("records")
        print(f"Resume {model_id}: {len(done_ids)} rows already done")
    else:
        done_ids = set()
        rows = []

    pipe = load_vlm(model_id)
    start_time = time.time()
    processed_since_save = 0

    try:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Refining with {model_id}"):
            item_id = str(row[ID_COLUMN])
            if item_id in done_ids:
                continue

            claim = str(row[TEXT_COLUMN]).strip()
            image_paths = resolve_image_paths(row[IMAGE_COLUMN], DATASET_ROOT)
            base = {
                ID_COLUMN: row[ID_COLUMN],
                TEXT_COLUMN: row[TEXT_COLUMN],
                IMAGE_COLUMN: row[IMAGE_COLUMN],
            }

            try:
                parsed, raw, error = generate_refine_json(pipe, claim, image_paths)
                flat = flatten_refine_result(parsed, model_id, raw, error)
            except Exception as exc:
                fallback = normalize_refine_json({}, claim, bool(image_paths))
                flat = flatten_refine_result(fallback, model_id, "", f"{type(exc).__name__}: {exc}")

            rows.append({**base, **flat})
            done_ids.add(item_id)
            processed_since_save += 1

            if processed_since_save >= SAVE_EVERY:
                write_refine_rows(rows, output_csv)
                processed_since_save = 0

        write_refine_rows(rows, output_csv)
    finally:
        unload_model(pipe)

    minutes = (time.time() - start_time) / 60
    print(f"Saved {output_csv} | total rows in file: {len(rows)} | elapsed minutes: {minutes:.1f}")


## 8. Chạy Qwen trước

Đây là model chính được khuyến nghị cho refine multimodal tiếng Việt trên A100 40GB.

In [ ]:
if RUN_QWEN:
    run_refinement(QWEN_MODEL_ID, QWEN_OUTPUT_CSV)
else:
    print("Skipping Qwen because RUN_QWEN=False")


## 9. Chạy Gemma sau

Nếu `google/gemma-3-12b-it` bị OOM hoặc chưa được cấp quyền, đổi `GEMMA_MODEL_ID` thành `google/gemma-3-4b-it`, rồi chạy lại cell này.

In [ ]:
if RUN_GEMMA:
    run_refinement(GEMMA_MODEL_ID, GEMMA_OUTPUT_CSV)
else:
    print("Skipping Gemma because RUN_GEMMA=False")


## 10. Gộp kết quả hai model

File gộp dùng dạng long format: mỗi claim có thể có hai dòng, một dòng Qwen và một dòng Gemma.

In [ ]:
parts = []
for path in [QWEN_OUTPUT_CSV, GEMMA_OUTPUT_CSV]:
    if path.exists():
        parts.append(pd.read_csv(path))

if parts:
    combined = pd.concat(parts, ignore_index=True)
    combined.to_csv(COMBINED_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    display(combined[[ID_COLUMN, "model_name", "refined_primary_retrieval_query", "refine_error"]].head())
    print(f"Saved combined CSV: {COMBINED_OUTPUT_CSV}")
else:
    print("No output CSV found yet.")


## 11. Kiểm tra nhanh lỗi format

Dùng cell này để xem tỉ lệ lỗi parse JSON hoặc lỗi inference theo từng model.

In [ ]:
if COMBINED_OUTPUT_CSV.exists():
    combined = pd.read_csv(COMBINED_OUTPUT_CSV)
    error_report = (
        combined.assign(has_error=combined["refine_error"].fillna("").str.len() > 0)
        .groupby("model_name")
        .agg(rows=(ID_COLUMN, "count"), errors=("has_error", "sum"))
        .reset_index()
    )
    error_report["error_rate"] = error_report["errors"] / error_report["rows"]
    display(error_report)
    display(combined.loc[combined["refine_error"].fillna("").str.len() > 0, [ID_COLUMN, "model_name", "refine_error", "raw_output"]].head(10))
else:
    print("Run the refinement cells first.")
